# 01 – Data Cleaning

This notebook covers loading the CKD dataset, inspecting data quality, handling missing values, and removing duplicates.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
TRAIN_PATH = '../dataset/Training_CKD_dataset.csv'
TEST_PATH  = '../dataset/Testing_CKD_dataset.csv'

# ── Load data ──────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print('Training set shape :', train_df.shape)
print('Testing  set shape :', test_df.shape)

Training set shape : (21000, 36)
Testing  set shape : (4800, 36)


# 1. First Look at the Data


In [4]:
train_df.head()

,Target,Age,Gender,BMI,Systolic_BP,Diastolic_BP,Heart_Rate,Serum_Creatinine,Blood_Urea_Nitrogen,eGFR,...,Fasting_Glucose,HbA1c,Cholesterol,Triglycerides,Serum_Albumin,Total_Protein,Diabetes,Hypertension,Smoking_Status,Family_History_Kidney
0,Healthy Kidney,29,1,28,97,69,99,0,12,95,...,96,7.547874,204,120,4,7.091259,Yes,Yes,Yes,Yes
1,Severe CKD (Stage 4),43,0,18,165,100,67,5,87,28,...,88,7.287338,166,277,2,7.875167,Yes,Yes,Yes,No
2,Healthy Kidney,77,0,32,116,63,101,0,16,100,...,82,9.114854,246,299,4,7.083558,No,No,Yes,No
3,Healthy Kidney,83,0,24,93,75,87,0,10,101,...,82,7.286450,173,285,4,6.428780,Yes,No,No,Yes
4,Healthy Kidney,38,1,19,111,70,92,0,10,102,...,106,8.376492,266,294,4,7.852894,Yes,No,Yes,No


In [5]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21000 entries, 0 to 20999
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Target                    21000 non-null  str    
 1   Age                       21000 non-null  int64  
 2   Gender                    21000 non-null  int64  
 3   BMI                       21000 non-null  int64  
 4   Systolic_BP               21000 non-null  int64  
 5   Diastolic_BP              21000 non-null  int64  
 6   Heart_Rate                21000 non-null  int64  
 7   Serum_Creatinine          21000 non-null  int64  
 8   Blood_Urea_Nitrogen       21000 non-null  int64  
 9   eGFR                      21000 non-null  int64  
 10  Urine_Albumin             21000 non-null  int64  
 11  Urine_Protein             21000 non-null  int64  
 12  Albumin_Creatinine_Ratio  21000 non-null  int64  
 13  Urine_Specific_Gravity    21000 non-null  float64
 14  Sodium           

In [6]:
train_df.describe().T.style.format('{:.3f}').background_gradient(cmap='Blues', axis=0)

,count,mean,std,min,25%,50%,75%,max
Age,21000.000,51.952,18.796,20.000,36.000,52.000,68.000,84.000
Gender,21000.000,0.501,0.500,0.000,0.000,1.000,1.000,1.000
BMI,21000.000,25.978,4.890,18.000,22.000,26.000,30.000,34.000
Systolic_BP,21000.000,113.498,19.152,90.000,99.000,110.000,120.000,189.000
Diastolic_BP,21000.000,75.281,12.107,60.000,66.000,73.000,80.000,119.000
Heart_Rate,21000.000,84.281,14.388,60.000,72.000,84.000,97.000,109.000
Serum_Creatinine,21000.000,0.630,1.482,0.000,0.000,0.000,1.000,9.000
Blood_Urea_Nitrogen,21000.000,21.682,20.800,7.000,11.000,15.000,20.000,149.000
eGFR,21000.000,91.426,26.787,5.000,89.000,99.000,109.000,119.000
Urine_Albumin,21000.000,59.987,136.149,0.000,6.000,13.000,20.000,999.000


## 2. Missing Values Analysis

In [7]:
# ── Missing value counts ───────────────────────────────────────────────────
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count':      missing,
    'Missing Percentage': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

if missing_report.empty:
    print('✅ No missing values found in the training set!')
else:
    display(missing_report)

✅ No missing values found in the training set!


In [8]:
# ── Visualise missing values ───────────────────────────────────────────────
total_missing = train_df.isnull().sum().sum()                  
print(f'Total missing cells : {total_missing}')
print(f'Total cells         : {train_df.size}')
print(f'Missing percentage  : {total_missing / train_df.size * 100:.4f}%')

Total missing cells : 0
Total cells         : 756000
Missing percentage  : 0.0000%


## 3. Duplicate Row Detection

In [9]:
dup_count = train_df.duplicated().sum()
print(f'Duplicate rows in training set: {dup_count}')

if dup_count > 0:
    train_df = train_df.drop_duplicates()
    print(f'Cleaned training set shape: {train_df.shape}')

Duplicate rows in training set: 0


In [10]:
cat_cols = train_df.select_dtypes(include='object').columns.tolist()
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'Numerical  columns  ({len(num_cols)}): {num_cols}')

Categorical columns (5): ['Target', 'Diabetes', 'Hypertension', 'Smoking_Status', 'Family_History_Kidney']
Numerical  columns  (31): ['Age', 'Gender', 'BMI', 'Systolic_BP', 'Diastolic_BP', 'Heart_Rate', 'Serum_Creatinine', 'Blood_Urea_Nitrogen', 'eGFR', 'Urine_Albumin', 'Urine_Protein', 'Albumin_Creatinine_Ratio', 'Urine_Specific_Gravity', 'Sodium', 'Potassium', 'Calcium', 'Phosphorus', 'Chloride', 'Bicarbonate', 'Hemoglobin', 'RBC_Count', 'WBC_Count', 'Platelet_Count', 'Packed_Cell_Volume', 'Blood_Glucose_Random', 'Fasting_Glucose', 'HbA1c', 'Cholesterol', 'Triglycerides', 'Serum_Albumin', 'Total_Protein']


In [11]:
# ── Unique values for each categorical column ─────────────────────────────
for col in cat_cols:
    print(f'{col:35s}: {train_df[col].unique()}')

Target                             : <ArrowStringArray>
[          'Healthy Kidney',     'Severe CKD (Stage 4)',
     'Mild CKD (Stage 1–2)',   'Moderate CKD (Stage 3)',
 'Kidney Failure (Stage 5)']
Length: 5, dtype: str
Diabetes                           : <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Hypertension                       : <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Smoking_Status                     : <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Family_History_Kidney              : <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str


## 5. Categorical Encoding

In [12]:
# ── Encode binary Yes/No columns ──────────────────────────────────────────
BINARY_COLS = ['Diabetes', 'Hypertension', 'Smoking_Status', 'Family_History_Kidney']

train_enc = train_df.copy()
for col in BINARY_COLS:
    if col in train_enc.columns:
        train_enc[col] = train_enc[col].map({'Yes': 1, 'No': 0})

# ── Encode the Target column ──────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_enc['Target_enc'] = le.fit_transform(train_enc['Target'])

print('Class mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

Class mapping:
  0 → Healthy Kidney
  1 → Kidney Failure (Stage 5)
  2 → Mild CKD (Stage 1–2)
  3 → Moderate CKD (Stage 3)
  4 → Severe CKD (Stage 4)


## 6. Data Type Summary After Cleaning

In [13]:
print('Data types after encoding:')
print(train_enc.dtypes)
print(f'\nFinal cleaned training shape: {train_enc.shape}')
print('\n✅ Data cleaning complete!')

Data types after encoding:
Target                          str
Age                           int64
Gender                        int64
BMI                           int64
Systolic_BP                   int64
Diastolic_BP                  int64
Heart_Rate                    int64
Serum_Creatinine              int64
Blood_Urea_Nitrogen           int64
eGFR                          int64
Urine_Albumin                 int64
Urine_Protein                 int64
Albumin_Creatinine_Ratio      int64
Urine_Specific_Gravity      float64
Sodium                        int64
Potassium                     int64
Calcium                       int64
Phosphorus                    int64
Chloride                      int64
Bicarbonate                   int64
Hemoglobin                    int64
RBC_Count                   float64
WBC_Count                     int64
Platelet_Count                int64
Packed_Cell_Volume            int64
Blood_Glucose_Random          int64
Fasting_Glucose               int64
H